# Legal QA Normalization

Chuẩn hóa dữ liệu QA pháp luật từ CSV/JSONL bằng LLM API.

**Input:** `question`, `answer`  
**Output:** `processed_answer`, `relevant_documents`, `relevant_laws`

Các field RAW khác được giữ nguyên. Notebook chạy trên máy local; CPU chỉ xử lý dữ liệu và kiểm tra kết quả, LLM inference thực hiện qua API.

## 1. Dependencies

In [1]:
%pip install -q --upgrade openai pandas tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuration

In [14]:

from pathlib import Path
import os
import json
import re
import time
import math
import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

from dotenv import load_dotenv

# load env
load_dotenv()



DATA_DIR = Path('../data').resolve()
print("DATA_DIR:", DATA_DIR)
RAW_DATA_DIR = DATA_DIR / "raw" 

RAW_QA_DIR = RAW_DATA_DIR / "legal_qa"
PROCESSED_QA_DIR = DATA_DIR / "processed_legal_qa"
INPUT_PATH = RAW_QA_DIR / "processed_all_qa.jsonl"  # .csv hoặc .jsonl
OUTPUT_DIR = PROCESSED_QA_DIR

MODEL_NAME = "gpt-6-luna"
QUESTION_FIELD = "question"
ANSWER_FIELD = "answer"

MAX_WORKERS = 2
MAX_RETRIES = 5
MAX_ROWS = None                 # None: xử lý toàn bộ
PRESERVE_RAW_FIELDS = True
DEDUPLICATE_BY_CONTENT_SHA256 = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.jsonl"
ERROR_PATH = OUTPUT_DIR / "normalization_errors.jsonl"
AUDIT_PATH = OUTPUT_DIR / "normalization_audit.jsonl"
FINAL_PATH = OUTPUT_DIR / "normalized_legal_qa.jsonl"


DATA_DIR: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend\data


## 3. API client

In [15]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    api_key = getpass.getpass("OPENAI_API_KEY: ")

client = OpenAI(api_key=api_key, timeout=120, max_retries=0)


## 4. Load input

In [16]:
def load_input(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path, encoding="utf-8-sig")
    elif suffix in {".jsonl", ".ndjson"}:
        records = []
        with path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSONL at line {line_no}: {exc}") from exc
                if not isinstance(record, dict):
                    raise ValueError(f"Line {line_no} must contain a JSON object.")
                records.append(record)
        df = pd.DataFrame(records)
    else:
        raise ValueError("Input format must be CSV, JSONL, or NDJSON.")

    missing = [x for x in [QUESTION_FIELD, ANSWER_FIELD] if x not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    return df


if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_PATH.resolve()}")

raw_df = load_input(INPUT_PATH)

if MAX_ROWS is not None:
    raw_df = raw_df.head(MAX_ROWS).copy()

raw_df = raw_df.reset_index(drop=True)
raw_df["_source_row_index"] = range(len(raw_df))

if DEDUPLICATE_BY_CONTENT_SHA256 and "content_sha256" in raw_df.columns:
    hashes = raw_df["content_sha256"].fillna("").astype(str).str.strip()
    duplicate_mask = hashes.ne("") & hashes.duplicated(keep="first")
    raw_df = raw_df.loc[~duplicate_mask].reset_index(drop=True)
    raw_df["_source_row_index"] = range(len(raw_df))

print("Rows:", len(raw_df))
print("Columns:", raw_df.columns.tolist())
display(raw_df.head(3))


Rows: 2234
Columns: ['id', 'url', 'source', 'category', 'title', 'question', 'answer', 'citation', 'citation_verified', 'related_legal_documents', 'lawyer', 'listing_page', 'extraction_status', 'content_sha256', 'collected_at_utc', 'topic', 'relevant_law', 'answer_url', 'source_category_url', '_source_row_index']


,id,url,source,category,title,question,answer,citation,citation_verified,related_legal_documents,lawyer,listing_page,extraction_status,content_sha256,collected_at_utc,topic,relevant_law,answer_url,source_category_url,_source_row_index
0,da62d3aa6f234b7d92ee,https://luatvietnam.vn/luat-su-tu-van/quyen-do...,LuatVietnam,Dân sự,Quyền đơn phương chấm dứt hợp đồng khi một bên...,Xin hỏi LuatVietnam: Tôi là một ca sỹ. Tôi đã ...,Trả lời:\n\nTrường hợp của bạn thể hiện Công t...,[Điều 428 Bộ luật Dân sự 2015 quy định về quyề...,False,"[{'title': '', 'url': 'https://luatvietnam.vn/...",Được tư vấn bởi: Luật sư Hoàng Tuấn Vũ Công t...,1.0,full_qa,c5f3a34ee72751b8da1b8371e72bcbb551b7e128459484...,2026-10-28T11:37:33.972299+00:00,Dân sự,None,NaN,NaN,0
1,91ddd56ddd285f23bba7,https://luatvietnam.vn/luat-su-tu-van/nguoi-kh...,LuatVietnam,Dân sự,Người khởi kiện vắng mặt lần thứ 2 thì có đình...,Xin hỏi LuatVietnam: Điểm c khoản 1 Điều 217 c...,Trả lời:\n\nCăn cứ Công văn 163/TANDTC-PC ngày...,"[Điều 217 của Bộ luật Tố tụng dân sự quy định,...",False,"[{'title': '', 'url': 'https://luatvietnam.vn/...",Được tư vấn bởi: Luật sư Hoàng Tuấn Vũ Công t...,1.0,full_qa,54ec8f4aeb8761b7c49ec1ac5c0aa33ae84433fdb46192...,2026-10-28T11:37:37.068104+00:00,Dân sự,None,NaN,NaN,1
2,2e3f08ddd0ff3501e0f1,https://luatvietnam.vn/luat-su-tu-van/nguoi-kh...,LuatVietnam,Dân sự,Người khởi kiện vắng mặt lần thứ 2 thì có đình...,Xin hỏi LuatVietnam: Điểm c khoản 1 Điều 217 c...,Trả lời:\n\nCăn cứ Công văn 163/TANDTC-PC ngày...,"[Điều 217 của Bộ luật Tố tụng dân sự quy định,...",False,[],Được tư vấn bởi: Luật sư Hoàng Tuấn Vũ Công t...,1.0,full_qa,54ec8f4aeb8761b7c49ec1ac5c0aa33ae84433fdb46192...,2026-10-28T11:37:40.334654+00:00,Dân sự,None,NaN,NaN,2


## 5. Prompt

In [17]:
SYSTEM_PROMPT = '''
Bạn là bộ xử lý dữ liệu hỏi đáp pháp luật Việt Nam.
Đầu vào gồm question và answer gốc. Chỉ trả về một JSON object hợp lệ.

1. processed_answer
- Viết lại answer ngắn gọn, rõ nghĩa.
- Giữ nguyên kết luận, điều kiện, ngoại lệ và các phân biệt quan trọng.
- Không chép dài nội dung điều luật; chỉ nhắc vị trí căn cứ nếu answer có nêu.
- Không thêm sự kiện, căn cứ hoặc kết luận mới.
- Giữ cách diễn đạt thận trọng nếu answer chưa đủ thông tin.

2. relevant_documents
- Liệt kê tên văn bản pháp luật được viện dẫn trong answer.
- Không tự thêm văn bản chỉ vì thấy có liên quan.
- Không đưa website hoặc nguồn tư vấn vào danh sách.

3. relevant_laws
Mỗi căn cứ pháp luật được viện dẫn trong answer là một object:
{
  "document_name": "Tên văn bản",
  "citation": "khoản 3 Điều 210 Bộ luật ...",
  "hierarchy": {
    "article": "210",
    "clause": "3",
    "point": null,
    "section": null,
    "subsection": null,
    "part": null,
    "chapter": null
  },
  "content": null,
  "evidence_in_raw_answer": "Đoạn nguyên văn trong answer"
}

Quy tắc:
- Chỉ trích căn cứ có thể tìm thấy trong answer.
- Chuẩn hóa citation nhưng không tự suy đoán cấp hierarchy.
- Chỉ điền cấp article, clause, point, section, subsection, part, chapter
  khi answer nêu rõ; cấp không xác định phải là null.
- Không suy ra khoản hoặc điểm từ kiến thức nội tại.
- content chỉ chứa nội dung quy phạm được chép trong answer.
  Giữ nguyên câu chữ. Nếu answer chỉ nêu vị trí điều luật hoặc diễn giải,
  content phải là null.
- evidence_in_raw_answer phải là đoạn liên tục, sao chép nguyên văn
  từ answer và thể hiện căn cứ được viện dẫn.
- Nếu có nhiều khoản hoặc điểm được viện dẫn riêng, tạo object riêng.
- Nếu không có căn cứ được viện dẫn, relevant_documents=[]
  và relevant_laws=[].

Cấu trúc JSON:
{
  "processed_answer": "string",
  "relevant_documents": ["string"],
  "relevant_laws": [
    {
      "document_name": "string",
      "citation": "string",
      "hierarchy": {
        "article": "string hoặc null",
        "clause": "string hoặc null",
        "point": "string hoặc null",
        "section": "string hoặc null",
        "subsection": "string hoặc null",
        "part": "string hoặc null",
        "chapter": "string hoặc null"
      },
      "content": "string hoặc null",
      "evidence_in_raw_answer": "string hoặc null"
    }
  ]
}

Không trả markdown hoặc giải thích ngoài JSON.
'''.strip()

HIERARCHY_FIELDS = [
    "article", "clause", "point", "section",
    "subsection", "part", "chapter"
]


## 6. Validation

In [18]:
def normalize_whitespace(value: str) -> str:
    return re.sub(r"\s+", " ", value or "").strip()


def contains_verbatim(piece: str, source: str) -> bool:
    piece_norm = normalize_whitespace(piece)
    source_norm = normalize_whitespace(source)
    return bool(piece_norm) and piece_norm in source_norm


def validate_result(result: dict, raw_answer: str) -> list[str]:
    errors = []

    if not isinstance(result.get("processed_answer"), str) or not result["processed_answer"].strip():
        errors.append("processed_answer must be a non-empty string")

    documents = result.get("relevant_documents")
    if not isinstance(documents, list) or not all(isinstance(x, str) for x in documents):
        errors.append("relevant_documents must be a list of strings")
        documents = []

    laws = result.get("relevant_laws")
    if not isinstance(laws, list):
        errors.append("relevant_laws must be a list")
        laws = []

    seen_citations = set()

    for i, law in enumerate(laws):
        prefix = f"relevant_laws[{i}]"

        if not isinstance(law, dict):
            errors.append(f"{prefix} must be an object")
            continue

        required = [
            "document_name", "citation", "hierarchy",
            "content", "evidence_in_raw_answer"
        ]
        for key in required:
            if key not in law:
                errors.append(f"{prefix} missing {key}")

        for key in ["document_name", "citation"]:
            if not isinstance(law.get(key), str) or not law[key].strip():
                errors.append(f"{prefix}.{key} must be a non-empty string")

        hierarchy = law.get("hierarchy")
        if not isinstance(hierarchy, dict):
            errors.append(f"{prefix}.hierarchy must be an object")
        else:
            for key in HIERARCHY_FIELDS:
                if key not in hierarchy:
                    errors.append(f"{prefix}.hierarchy missing {key}")
                elif hierarchy[key] is not None and not isinstance(hierarchy[key], str):
                    errors.append(f"{prefix}.hierarchy.{key} must be string or null")

        evidence = law.get("evidence_in_raw_answer")
        if not isinstance(evidence, str) or not evidence.strip():
            errors.append(f"{prefix}.evidence_in_raw_answer is empty")
        elif not contains_verbatim(evidence, raw_answer):
            errors.append(f"{prefix}.evidence_in_raw_answer not found verbatim in answer")

        content = law.get("content")
        if content is not None:
            if not isinstance(content, str):
                errors.append(f"{prefix}.content must be string or null")
            elif not contains_verbatim(content, raw_answer):
                errors.append(f"{prefix}.content not found verbatim in answer")

        doc_name = law.get("document_name")
        if isinstance(doc_name, str) and doc_name and doc_name not in documents:
            errors.append(f"{prefix}.document_name missing from relevant_documents")

        citation = normalize_whitespace(str(law.get("citation", ""))).lower()
        if citation and citation in seen_citations:
            errors.append(f"{prefix}.citation is duplicated")
        seen_citations.add(citation)

    return errors


## 7. LLM request

In [19]:
def call_model(question: str, answer: str) -> tuple[dict, dict]:
    payload = {"question": question, "answer": answer}

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False)
            }
        ]
    )

    content = response.choices[0].message.content
    if not content:
        raise ValueError("Empty model response")

    result = json.loads(content)
    usage = response.usage

    token_usage = {
        "prompt_tokens": getattr(usage, "prompt_tokens", None) if usage else None,
        "completion_tokens": getattr(usage, "completion_tokens", None) if usage else None,
        "total_tokens": getattr(usage, "total_tokens", None) if usage else None
    }

    return result, token_usage


## 8. Smoke test

In [20]:
test_answer = "Theo khoản 3 Điều 10 Văn bản thử nghiệm: Bên A phải thông báo cho bên B."
test_result = {
    "processed_answer": "Bên A phải thông báo theo khoản 3 Điều 10 Văn bản thử nghiệm.",
    "relevant_documents": ["Văn bản thử nghiệm"],
    "relevant_laws": [{
        "document_name": "Văn bản thử nghiệm",
        "citation": "khoản 3 Điều 10 Văn bản thử nghiệm",
        "hierarchy": {
            "article": "10", "clause": "3", "point": None,
            "section": None, "subsection": None, "part": None, "chapter": None
        },
        "content": "Bên A phải thông báo cho bên B.",
        "evidence_in_raw_answer": "Theo khoản 3 Điều 10 Văn bản thử nghiệm:"
    }]
}

assert validate_result(test_result, test_answer) == []
print("Validation smoke test: PASS")


Validation smoke test: PASS


## 9. Process one record

In [21]:
def process_record(row_index: int, row: dict) -> dict:
    question = "" if pd.isna(row.get(QUESTION_FIELD)) else str(row.get(QUESTION_FIELD))
    answer = "" if pd.isna(row.get(ANSWER_FIELD)) else str(row.get(ANSWER_FIELD))

    if not question.strip() or not answer.strip():
        return {
            "status": "error",
            "source_row_index": row_index,
            "error": "question or answer is empty"
        }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result, usage = call_model(question, answer)

            required = {"processed_answer", "relevant_documents", "relevant_laws"}
            missing = required - result.keys()
            if missing:
                raise ValueError(f"Missing output fields: {sorted(missing)}")

            errors = validate_result(result, answer)
            if errors:
                return {
                    "status": "error",
                    "source_row_index": row_index,
                    "error": " | ".join(errors),
                    "usage": usage
                }

            output = row.copy() if PRESERVE_RAW_FIELDS else {
                QUESTION_FIELD: question,
                ANSWER_FIELD: answer
            }

            output[QUESTION_FIELD] = question
            output[ANSWER_FIELD] = answer
            output["processed_answer"] = result["processed_answer"]
            output["relevant_documents"] = result["relevant_documents"]
            output["relevant_laws"] = result["relevant_laws"]

            return {
                "status": "success",
                "source_row_index": row_index,
                "output": output,
                "usage": usage
            }

        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            if attempt < MAX_RETRIES:
                time.sleep(min(2 ** (attempt - 1), 30))

    return {
        "status": "error",
        "source_row_index": row_index,
        "error": last_error
    }


## 10. Process sample

In [22]:
sample_row = None
sample_index = None

for index, row in raw_df.iterrows():
    question = "" if pd.isna(row.get(QUESTION_FIELD)) else str(row.get(QUESTION_FIELD))
    answer = "" if pd.isna(row.get(ANSWER_FIELD)) else str(row.get(ANSWER_FIELD))
    if question.strip() and answer.strip():
        sample_row = row.drop(labels=["_source_row_index"]).to_dict()
        sample_index = int(index)
        break

if sample_row is None:
    raise ValueError("No record has both question and answer")

sample_output = process_record(sample_index, sample_row)
print(json.dumps(sample_output, ensure_ascii=False, indent=2, default=str))


{
  "status": "error",
  "source_row_index": 0,
  "error": "RateLimitError: Error code: 429 - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/."
}


## 11. Checkpoint functions

In [23]:
file_lock = Lock()


def append_jsonl(path: Path, record: dict) -> None:
    with file_lock:
        with path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def read_checkpoint(path: Path) -> dict[int, dict]:
    latest = {}
    if not path.exists():
        return latest

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                record = json.loads(line)
                latest[int(record["source_row_index"])] = record
            except (json.JSONDecodeError, KeyError, ValueError, TypeError):
                continue

    return latest


## 12. Batch processing

In [24]:
latest_checkpoint = read_checkpoint(CHECKPOINT_PATH)
completed_indices = {
    index for index, record in latest_checkpoint.items()
    if record.get("status") == "success"
}

pending = []
for index, row in raw_df.iterrows():
    if int(index) in completed_indices:
        continue
    row_data = row.drop(labels=["_source_row_index"]).to_dict()
    pending.append((int(index), row_data))

print("Completed:", len(completed_indices))
print("Pending:", len(pending))


Completed: 0
Pending: 2234


In [25]:
success_count = 0
error_count = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_record, index, row): index
        for index, row in pending
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Normalize QA"):
        result = future.result()
        append_jsonl(CHECKPOINT_PATH, result)
        latest_checkpoint[result["source_row_index"]] = result

        if result["status"] == "success":
            success_count += 1
        else:
            error_count += 1
            append_jsonl(ERROR_PATH, result)

print("Success:", success_count)
print("Errors:", error_count)


Normalize QA:   0%|          | 6/2234 [00:49<4:13:55,  6.84s/it] 

: 

: 

## 13. Export normalized data and audit

In [ ]:
success_records = []
error_records = []
audit_records = []

for index in sorted(latest_checkpoint):
    record = latest_checkpoint[index]
    if record.get("status") == "success":
        success_records.append(record["output"])
        audit_records.append({
            "source_row_index": index,
            "usage": record.get("usage", {})
        })
    else:
        error_records.append({
            "source_row_index": index,
            "error": record.get("error")
        })


def write_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


write_jsonl(FINAL_PATH, success_records)
write_jsonl(ERROR_PATH, error_records)
write_jsonl(AUDIT_PATH, audit_records)

print("Normalized records:", len(success_records))
print("Error records:", len(error_records))
print("Output:", FINAL_PATH.resolve())


## 14. Token usage and sample review

In [ ]:
token_totals = {
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "total_tokens": 0
}

for record in success_records:
    pass

for audit in audit_records:
    usage = audit.get("usage") or {}
    for key in token_totals:
        token_totals[key] += usage.get(key) or 0

print("Token usage:", token_totals)

for i, record in enumerate(success_records[:5], start=1):
    print(f"--- SAMPLE {i} ---")
    print("Question:", record.get("question", ""))
    print("Processed answer:", record.get("processed_answer", ""))
    print(json.dumps(record.get("relevant_laws", []), ensure_ascii=False, indent=2))


## 15. Validation scope

- Kiểm tra cấu trúc JSON và kiểu dữ liệu.
- Kiểm tra `evidence_in_raw_answer` và `content` có xuất hiện trong `answer`.
- Kiểm tra các cấp `hierarchy` và citation trùng lặp.

Các kiểm tra này không xác minh tính đúng đắn, tính đầy đủ hoặc hiệu lực của căn cứ pháp luật. Cần đối chiếu riêng với corpus pháp luật chính thức và kiểm duyệt các mẫu được dùng để huấn luyện.